**Emotion** detection using Deep **learning**

import libraries


In [ ]:
!pip install librosa soundfile

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

loading audio dataset/files & extracting it


In [ ]:
!wget https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
!unzip Audio_Speech_Actors_01-24.zip

--2026-02-21 04:41:57--  https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.185.48.75, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/1188976/files/Audio_Speech_Actors_01-24.zip [following]
--2026-02-21 04:41:57--  https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 208468073 (199M) [application/octet-stream]
Saving to: ‘Audio_Speech_Actors_01-24.zip’

Audio_Speech_Actors 100%[===================>] 198.81M  25.2MB/s    in 8.9s    

2026-02-21 04:42:07 (22.2 MB/s) - ‘Audio_Speech_Actors_01-24.zip’ saved [208468073/208468073]

Archive:  Audio_Speech_Actors_01-24.zip
   creating: Actor_01/
  inflating: Actor_01/03-01-01-01-01-01-01.wav  
  inflating: Actor_01/03-01-01-01

In [ ]:
import librosa
import numpy as np
import os

exacting mfcc features

In [ ]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path, duration=3, offset=0.5)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)

    if mfcc.shape[1] < 174:
        pad_width = 174 - mfcc.shape[1]
        mfcc = np.pad(mfcc, pad_width=((0,0),(0,pad_width)))
    else:
        mfcc = mfcc[:, :174]

    return mfcc

In [ ]:
!ls

Actor_01  Actor_07  Actor_13  Actor_19	Audio_Speech_Actors_01-24.zip
Actor_02  Actor_08  Actor_14  Actor_20	sample_data
Actor_03  Actor_09  Actor_15  Actor_21
Actor_04  Actor_10  Actor_16  Actor_22
Actor_05  Actor_11  Actor_17  Actor_23
Actor_06  Actor_12  Actor_18  Actor_24


extacting feature for one audio file

In [ ]:
sample_file = "Actor_01/03-01-01-01-01-01-01.wav"
features = extract_features(sample_file)

print(features.shape)

(40, 174)


extracting features for entire dataset

In [ ]:
features = []
labels = []

emotion_map = {
    "01":"neutral",
    "02":"calm",
    "03":"happy",
    "04":"sad",
    "05":"angry",
    "06":"fearful",
    "07":"disgust",
    "08":"surprised"
}

for actor in os.listdir():
    if actor.startswith("Actor"):
        for file in os.listdir(actor):
            file_path = os.path.join(actor, file)

            mfcc = extract_features(file_path)
            features.append(mfcc)

            emotion = file.split("-")[2]
            labels.append(emotion_map[emotion])

print("Total samples:", len(features))

Total samples: 1440


In [ ]:
X = np.array(features)
X = X[..., np.newaxis]

In [ ]:
print(X.shape)

(1440, 40, 174, 1)


preparing data for model (no text labels)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

encoder = LabelEncoder()
y = encoder.fit_transform(labels)
y = to_categorical(y)

split Train/Test (dataset ready for training)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Build first model (simple neural network)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(40,174,1)),
    MaxPooling2D((2,2)),

    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128,activation="relu"),
    Dropout(0.3),
    Dense(8,activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 38, 172, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 19, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 17, 84, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 21504)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │     2,752,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,772,488 (10.58 MB)

 Trainable params: 2,772,488 (10.58 MB)

 Non-trainable params: 0 (0.00 B)

Training model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=12,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 11s 310ms/step - accuracy: 0.9695 - loss: 0.0920 - val_accuracy: 0.5868 - val_loss: 1.8750
Epoch 2/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 20s 307ms/step - accuracy: 0.9744 - loss: 0.0800 - val_accuracy: 0.6007 - val_loss: 1.9108
Epoch 3/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 11s 316ms/step - accuracy: 0.9690 - loss: 0.0855 - val_accuracy: 0.5764 - val_loss: 2.2086
Epoch 4/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 263ms/step - accuracy: 0.9645 - loss: 0.0861 - val_accuracy: 0.5799 - val_loss: 2.0857
Epoch 5/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 11s 305ms/step - accuracy: 0.9828 - loss: 0.0679 - val_accuracy: 0.5799 - val_loss: 1.9926
Epoch 6/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 11s 298ms/step - accuracy: 0.9788 - loss: 0.0658 - val_accuracy: 0.5799 - val_loss: 2.0988
Epoch 7/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 329ms/step - accuracy: 0.9787 - loss: 0.0457 - val_accuracy: 0.5833 - val_loss: 1.9594
Epoch 8/12
36/36 ━━━━━━━━━━━━━━━━━━━━ 11s 309ms/step - accuracy: 0.9819 - loss: 0.0538 - val_accur

Test Accuracy check

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test accuracy:", acc)

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.6123 - loss: 2.1054
Test accuracy: 0.59375


Save Model

In [ ]:
model.save("emotion_model.h5")

Convert to TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("emotion_model.tflite","wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpzn5m2esb'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 174, 1), dtype=tf.float32, name='keras_tensor_6')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  133021758680592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133021755532560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133021758670032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133021758680784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133021761322064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133021761321680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133021761320720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133021761319952: TensorSpec(shape=(), dtype=tf.resource, name=None)


Measure Normal Model Latency

In [ ]:
import time

sample = X_test[0:1]

start = time.time()
model.predict(sample)
end = time.time()

print("Normal model latency:", end-start)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Normal model latency: 0.1372079849243164


Measure TFLite Latency

In [ ]:
import numpy as np
import tensorflow as tf
import time

interpreter = tf.lite.Interpreter(model_path="emotion_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test[0:1].astype(np.float32)

start = time.time()

interpreter.set_tensor(input_details[0]['index'], sample)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])

end = time.time()

print("TFLite latency:", end-start)

TFLite latency: 0.007042884826660156


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Emotion Prediction Demo

In [ ]:
pred = model.predict(X_test[1:2])
emotion_index = np.argmax(pred)
print("Predicted emotion:", encoder.inverse_transform([emotion_index])[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Predicted emotion: sad


Checking with my voice

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving my_voice.mp4 to my_voice.mp4


Converting Mp4--> Wav

In [ ]:
!pip install moviepy

In [ ]:
file_name = list(uploaded.keys())[0]

In [ ]:
!ffmpeg -i "$file_name" -ar 16000 -ac 1 converted.wav -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
mfcc = extract_features("converted.wav")
mfcc = np.array(mfcc)
mfcc = mfcc[np.newaxis, ..., np.newaxis]

pred = model.predict(mfcc)
emotion_index = np.argmax(pred)

print("Predicted emotion:",
      encoder.inverse_transform([emotion_index])[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
Predicted emotion: disgust


In [ ]:
sample = X_test[5:6]
pred = model.predict(sample)
print(encoder.inverse_transform([np.argmax(pred)])[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
angry


In [ ]:
from google.colab import files
files.download("emotion_model.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp emotion_model.h5 /content/drive/MyDrive/
!cp emotion_model.tflite /content/drive/MyDrive/

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/emotion_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
!ls -lh emotion_model.h5

-rw-r--r-- 1 root root 32M Feb 21 05:35 emotion_model.h5


In [49]:
!zip model.zip emotion_model.h5 emotion_model.tflite

  adding: emotion_model.h5 (deflated 23%)
  adding: emotion_model.tflite (deflated 14%)


In [50]:
from google.colab import files
files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>